# Overview

This figure compares NRMSE across input data types, faceted by
target-variable family (linear vs log-transformed). Two panels keep the
bar count manageable and make the scale difference between families
visible. :scope: paper :figure: 2

## Data source

``` example
analysis/overall_performance.csv
```

# Setup

``` python
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
from neural_spd.config import PROJECT_ROOT
from neural_spd import plot_styles
from neural_spd.plot_styles import cm
plot_styles.apply()
PERF_METRICS_PATH = PROJECT_ROOT / "analysis/overall_performance.csv"
target_names = {
    'DoK': "$D/K$",
    'KoD': "$K/D$",
    'logDoK': r"$\log_{10}(D/K)$",
    'logKoD': r"$\log_{10}(K/D)$"
}
data_names = {
    'elevation': "Elevation",
    'flowacc': "Flow Acc.",
    'log10flowacc': r"$\log_{10}$(Flow Acc.)",
    'slope': "Slope",
    'curvature': "Curvature"
}
# order for x-axis: derived features first, then raw
_DATA_ORDER = ["Slope", "Curvature", r"$\log_{10}$(Flow Acc.)", "Flow Acc.", "Elevation"]
```

# Load data

``` python
perf_metrics_df = pd.read_csv(PERF_METRICS_PATH)
perf_metrics_df["target"] = perf_metrics_df["target"].map(target_names)
perf_metrics_df["data"] = perf_metrics_df["data"].map(data_names)

# Split into linear and log-transformed target families
linear_df = perf_metrics_df[perf_metrics_df["target"].isin(["$D/K$", "$K/D$"])]
log_df    = perf_metrics_df[perf_metrics_df["target"].isin(
    [r"$\log_{10}(D/K)$", r"$\log_{10}(K/D)$"])]
```

# Plotting function

``` python
_LIN_HUE   = ["$D/K$", "$K/D$"]
_LOG_HUE   = [r"$\log_{10}(D/K)$", r"$\log_{10}(K/D)$"]
_LIN_PAL   = [plot_styles.COLORS["green_dark"], plot_styles.COLORS["blue"]]
_LOG_PAL   = [plot_styles.COLORS["green_dark"], plot_styles.COLORS["blue"]]

def _bar_panel(ax, df, hue_order, palette, title):
    sns.barplot(
        data=df,
        x="data", y="nrmse", hue="target",
        hue_order=hue_order,
        order=_DATA_ORDER,
        ax=ax,
        palette=palette,
        edgecolor="none",
        capsize=0.05,
        err_kws={"linewidth": 0.8},
    )
    ax.set_ylabel("NRMSE")
    ax.set_xlabel("")
    ax.set_title(title)
    ax.legend(title="Target", loc="upper right")
    ax.tick_params(axis="x", labelrotation=25)
    for label in ax.get_xticklabels():
        label.set_ha("right")

def plot_perf_comp(axs):
    _bar_panel(axs[0], linear_df, _LIN_HUE, _LIN_PAL, "Linear targets")
    _bar_panel(axs[1], log_df,    _LOG_HUE,  _LOG_PAL, "Log-transformed targets")
    # panel labels
    for ax, letter in zip(axs, "ab"):
        plot_styles.panel_label(ax, letter, x=-0.08)
```

# Generate plots

``` python
plot_styles.double_column()
fig, ax = plt.subplots(figsize=(17*cm, 8.5*cm))
plot_perf_comp(ax)
plot_styles.save_figure(fig, "perf_compare", PROJECT_ROOT / "paper" / "figs")
fig.show()
```